In [1]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(invalid='ignore', divide='ignore')

from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath(".."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math
import pickle
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit

# never wrap a wide DataFrame across multiple lines
pd.set_option('display.expand_frame_repr', False)
# show all columns
pd.set_option('display.max_columns', None)
# pretend the display is super wide
pd.set_option('display.width', 10000)

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

def extract(ticker, returns, lb, cat_cols_all, windows=[10, 25]):

    import os
    # point this at an existing directory in your project
    os.chdir('/Users/brettchase/Documents/Fracturion/Feature_Set_Construction_Testing')
    
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

lb = 8
returns = [5, 10, 15, 20, 25, 35, 45]
raw_all = tags['Indicator'][tags['Type'] == 'Raw'].tolist()
df = extract('QQQ', returns, lb, raw_all, windows=[10,25])

# By category × velocity
duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
lag_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

def add_cyclic_seasonality(
    df: pd.DataFrame,
    date_col: str = 'Date',
    add_weekly: bool = True,
    add_month: bool = True,
    add_quarter: bool = True,
    add_year: bool = True,
    add_day_of_year: bool = False,
    mode: str = 'calendar',          # 'trading', 'calendar', or 'both'
    prefix: str = 'cyc_'
) -> pd.DataFrame:
    """
    Add cyclic (sin/cos) seasonality features.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a date column.
    date_col : str
        Name of the datetime column.
    add_weekly, add_month, add_quarter, add_year, add_day_of_year : bool
        Toggles for which seasonal cycles to create.
    mode : {'trading','calendar','both'}
        - 'trading': position = trading-day index / trading-days-in-period (resets at dataset slice start)
        - 'calendar': position = calendar progress within period (day-based; retains true mid-period phase)
        - 'both': create both sets (with suffixes _trade and _cal)
    prefix : str
        Prefix for created feature names.

    Returns
    -------
    pd.DataFrame
        Copy of df with new seasonality features appended (original row order preserved).
    """
    if date_col not in df.columns:
        raise ValueError(f"'{date_col}' not found in DataFrame.")
    if mode not in {'trading', 'calendar', 'both'}:
        raise ValueError("mode must be 'trading', 'calendar', or 'both'.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Preserve original order / index
    original_index = out.index
    work = out.sort_values(date_col).reset_index(drop=False)
    idx_col = work.columns[0]

    # Precompute basic date parts
    work['_year']    = work[date_col].dt.year
    work['_month']   = work[date_col].dt.month
    work['_quarter'] = work[date_col].dt.quarter
    work['_weekday'] = work[date_col].dt.weekday
    work['_doy']     = work[date_col].dt.dayofyear
    work['_is_leap'] = work[date_col].dt.is_leap_year

    # ---------- WEEKLY (trading week: Mon-Fri) ----------
    if add_weekly:
        week_pos = work['_weekday'] / 5.0  # Monday=0.0, Friday≈0.8 (if only Mon-Fri present)
        work[f'{prefix}week_sin'] = np.sin(2 * np.pi * week_pos)
        work[f'{prefix}week_cos'] = np.cos(2 * np.pi * week_pos)

    # Helper to safely divide
    def _safe_div(num, den):
        return np.where(den == 0, 0.0, num / np.where(den == 0, 1, den))

    # ---------- MONTH ----------
    if add_month:
        if mode in {'trading', 'both'}:
            grp_m = [work['_year'], work['_month']]
            idx_m = work.groupby(grp_m).cumcount()
            cnt_m = work.groupby(grp_m)[date_col].transform('count')
            pos_m_trade = _safe_div(idx_m, cnt_m)
            work[f'{prefix}month_trade_sin'] = np.sin(2 * np.pi * pos_m_trade)
            work[f'{prefix}month_trade_cos'] = np.cos(2 * np.pi * pos_m_trade)

        if mode in {'calendar', 'both'}:
            dom = work[date_col].dt.day           # 1..days_in_month
            dim = work[date_col].dt.days_in_month
            pos_m_cal = _safe_div(dom - 1, dim - 1)
            work[f'{prefix}month_sin'] = np.sin(2 * np.pi * pos_m_cal)
            work[f'{prefix}month_cos'] = np.cos(2 * np.pi * pos_m_cal)

    # ---------- QUARTER ----------
    if add_quarter:
        if mode in {'trading', 'both'}:
            grp_q = [work['_year'], work['_quarter']]
            idx_q = work.groupby(grp_q).cumcount()
            cnt_q = work.groupby(grp_q)[date_col].transform('count')
            pos_q_trade = _safe_div(idx_q, cnt_q)
            work[f'{prefix}quarter_trade_sin'] = np.sin(2 * np.pi * pos_q_trade)
            work[f'{prefix}quarter_trade_cos'] = np.cos(2 * np.pi * pos_q_trade)

        if mode in {'calendar', 'both'}:
            q = work['_quarter']
            y = work['_year']
            quarter_start = pd.to_datetime({'year': y, 'month': (q - 1) * 3 + 1, 'day': 1})
            quarter_end = quarter_start + pd.offsets.QuarterEnd(0)
            day_in_q = (work[date_col] - quarter_start).dt.days
            days_q = (quarter_end - quarter_start).dt.days
            pos_q_cal = _safe_div(day_in_q, days_q)
            work[f'{prefix}quarter_sin'] = np.sin(2 * np.pi * pos_q_cal)
            work[f'{prefix}quarter_cos'] = np.cos(2 * np.pi * pos_q_cal)

    # ---------- YEAR ----------
    if add_year:
        if mode in {'trading', 'both'}:
            grp_y = work['_year']
            idx_y = work.groupby(grp_y).cumcount()
            cnt_y = work.groupby(grp_y)[date_col].transform('count')
            pos_y_trade = _safe_div(idx_y, cnt_y)
            work[f'{prefix}year_trade_sin'] = np.sin(2 * np.pi * pos_y_trade)
            work[f'{prefix}year_trade_cos'] = np.cos(2 * np.pi * pos_y_trade)

        if mode in {'calendar', 'both'}:
            doy = work['_doy']
            year_len = np.where(work['_is_leap'], 366, 365)
            pos_y_cal = _safe_div(doy - 1, year_len - 1)
            work[f'{prefix}year_sin'] = np.sin(2 * np.pi * pos_y_cal)
            work[f'{prefix}year_cos'] = np.cos(2 * np.pi * pos_y_cal)

    # ---------- DAY OF YEAR (explicit) ----------
    if add_day_of_year:
        year_len = np.where(work['_is_leap'], 366, 365)
        work[f'{prefix}doy_sin'] = np.sin(2 * np.pi * work['_doy'] / year_len)
        work[f'{prefix}doy_cos'] = np.cos(2 * np.pi * work['_doy'] / year_len)

    # Drop temp columns
    temp_cols = [c for c in work.columns if c.startswith('_')]
    work = work.drop(columns=temp_cols)

    # Identify newly created features
    new_cols = [c for c in work.columns if c not in out.columns and c != idx_col]

    # Map back to original order
    work = work.set_index(idx_col)
    out[new_cols] = work.loc[original_index, new_cols]

    return out

weekly = ['cyc_week_sin', 'cyc_week_cos']
monthly = ['cyc_month_sin', 'cyc_month_cos']
quarterly = ['cyc_quarter_sin', 'cyc_quarter_cos']
yearly = ['cyc_year_sin', 'cyc_year_cos']
all_seasons = ['cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos',
               'cyc_year_sin', 'cyc_year_cos']
slopes = ['Close_slope10', 'Close_slope25', 'Close_slope50']

ticker = 'QQQ'
thresh = [.5]
results = []
returns = [2, 3, 4, 5, 8, 10, 15, 20]
all_perm_dfs = []
arch_types = ['deep']
lb = 9

with open("curr_feature_map.pkl", "rb") as f:
        feature_map = pickle.load(f)

selected_combos = {
    "h_5": {
        'lag_fast+trend_ratio_slow+momentum_moderate': lag_fast+trend_ratio_slow+momentum_moderate,
        'lag_fast+trend_slow+momentum_moderate': lag_fast+trend_slow+momentum_moderate,
        'duration_slow+trend_slow+momentum_moderate': duration_slow+trend_slow+momentum_moderate,
        'duration_fast+trend_slow+momentum_moderate': duration_fast+trend_slow+momentum_moderate,
        'lag_fast+trend_moderate+trend_ratio_slow': lag_fast+trend_moderate+trend_ratio_slow,
    },
    "h_10": {
        'lag_fast+duration_slow+momentum_moderate': lag_fast+duration_slow+momentum_moderate,
        'lag_fast+duration_fast+momentum_moderate': lag_fast+duration_fast+momentum_moderate,
        'lag_fast+trend_fast+momentum_moderate': lag_fast+trend_fast+momentum_moderate,
        'trend_slow+trend_ratio_slow+trend_ratio_fast': trend_slow+trend_ratio_slow+trend_ratio_fast,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
    },
    "h_15": {
        'duration_fast+trend_slow+trend_ratio_slow': duration_fast+trend_slow+trend_ratio_slow,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+trend_ratio_slow+trend_ratio_moderate': lag_fast+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+duration_slow+trend_slow': lag_fast+duration_slow+trend_slow,
        'duration_slow+trend_ratio_moderate+trend_ratio_fast': duration_slow+trend_ratio_moderate+trend_ratio_fast,
    },
    "h_20": {
        'trend_slow+trend_ratio_slow+momentum_moderate': trend_slow+trend_ratio_slow+momentum_moderate,
        'lag_fast+duration_slow+trend_ratio_moderate': lag_fast+duration_slow+trend_ratio_moderate,
        'lag_fast+duration_slow+trend_slow': lag_fast+duration_slow+trend_slow,
        'duration_slow+trend_ratio_moderate+volatility_moderate': duration_slow+trend_ratio_moderate+volatility_moderate,
        'lag_fast+trend_ratio_fast+momentum_moderate': lag_fast+trend_ratio_fast+momentum_moderate,
    },
    "h_25": {
        'duration_slow+trend_slow+trend_ratio_moderate': duration_slow+trend_slow+trend_ratio_moderate,
        'duration_slow+duration_fast+trend_ratio_moderate': duration_slow+duration_fast+trend_ratio_moderate,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+trend_ratio_moderate+volatility_moderate': lag_fast+trend_ratio_moderate+volatility_moderate,
        'duration_slow+trend_moderate+momentum_slow': duration_slow+trend_moderate+momentum_slow,
    },
    "h_35": {
        'duration_slow+duration_moderate+all_seasons': duration_slow+duration_moderate+all_seasons,
        'duration_slow+duration_moderate+momentum_slow': duration_slow+duration_moderate+momentum_slow,
        'duration_slow+momentum_slow+momentum_moderate': duration_slow+momentum_slow+momentum_moderate,
        'duration_slow+duration_fast+momentum_slow': duration_slow+duration_fast+momentum_slow,
        'trend_slow+trend_moderate+trend_ratio_fast': trend_slow+trend_moderate+trend_ratio_fast,
    },
    "h_45": {
        'duration_slow+duration_moderate+momentum_slow': duration_slow+duration_moderate+momentum_slow,
        'duration_slow+momentum_slow+momentum_moderate': duration_slow+momentum_slow+momentum_moderate,
        'duration_slow+duration_moderate+all_seasons': duration_slow+duration_moderate+all_seasons,
        'momentum_moderate+momentum_fast+all_seasons': momentum_moderate+momentum_fast+all_seasons,
        'duration_moderate+momentum_fast+all_seasons': duration_moderate+momentum_fast+all_seasons,
    }
}

def get_model_set(horizon):
    if horizon in [5]:
        key = "h_5"
    elif horizon in [10]:
        key = "h_10"
    elif horizon in [15]:
        key = "h_15"
    elif horizon in [20]:
        key = "h_20" 
    elif horizon in [25]:
        key = "h_25"
    elif horizon in [35]:
        key = "h_35"
    elif horizon in [45]:
        key = "h_45"
    else:
        raise ValueError(f"Unsupported horizon: {horizon}")
    
    return selected_combos[key]

model_dir = f'../Models/Ensemble_{ticker}'
thresh = [.5]

# Thresholds for prediction decision
upper = thresh[0]
lower = 1 - upper

# Placeholder for final predictions
combined_predictions = {}
df = extract(ticker, returns, lb, raw_all, windows=[10,25])
df = add_cyclic_seasonality(df, 
                            date_col='Date',
                            add_weekly=True,
                            add_month=True,
                            add_quarter=True,
                            add_year=True,
                            add_day_of_year=False,
                            mode='calendar',    # or 'trading' or 'both'
                            prefix='cyc_')

# Iterate through each return horizon
for r in returns:

    return_col = f"Return_{r}"

    df_ph = df.copy()

    # Create base prediction matrix with Date and Close
    pred_matrix = df_ph[['Date', 'Close']].copy()

    for arch in arch_types:
    
        selected_models = get_model_set(r)
            
        # all index-combos of length r
        for name, original_cols in selected_models.items():
            
            cols = feature_map.get((name, arch, r), original_cols)
            df_features = df_ph[cols].copy()
            df_features = df_features.replace([np.inf, -np.inf], 0)

            # XGBoost prediction
            xgb_path = os.path.join(model_dir, f"{name}_xgboost_{r}{arch}pi.pkl")
            if os.path.exists(xgb_path):
                with open(xgb_path, 'rb') as f:
                    model_xgb = pickle.load(f)
                prob_xgb = model_xgb.predict_proba(df_features)[:, 1]
                col_name_xgb = f"{name}_xgb_{r}{arch}"
                pred_matrix[col_name_xgb] = np.where(
                    prob_xgb > upper, prob_xgb,
                    np.where(prob_xgb < lower, prob_xgb - 1, 0)
                )

            # Final summed prediction across models
            pred_cols = [col for col in pred_matrix.columns if col not in ['Date', 'Close'] and not col.startswith('sum_')]
            pred_matrix[f"sum_{r}"] = pred_matrix[pred_cols].sum(axis=1)

            combined_predictions[r] = pred_matrix

aggr_df = None

for r, df_preds in combined_predictions.items():
    cols_to_merge = [col for col in df_preds.columns if col not in ['Date', 'Close']]
    df_preds_renamed = df_preds[['Date', 'Close'] + cols_to_merge].copy()

    if aggr_df is None:
        aggr_df = df_preds_renamed
    else:
        aggr_df = pd.merge(aggr_df, df_preds_renamed, on=['Date', 'Close'], how='outer')

sub_model_df = aggr_df.copy()
sub_model_df[['Date', 'Close', 'sum_2', 'sum_3', 'sum_4', 'sum_5', 'sum_8', 'sum_10',
              'sum_15', 'sum_20']].sort_index(ascending=False).head(21).round(3)

,Date,Close,sum_2,sum_3,sum_4,sum_5,sum_8,sum_10,sum_15,sum_20
1807,2025-09-24,595.670,-1.235,-2.301,-2.264,-2.200,-2.372,-0.183,-1.334,-2.160
1806,2025-09-23,598.200,-2.274,-2.325,-2.340,-2.223,-2.385,-0.157,-1.217,-1.136
1805,2025-09-22,602.200,-1.293,-1.373,-2.368,-1.224,-2.466,-0.067,-1.191,-1.051
1804,2025-09-19,598.656,-1.296,-1.332,-2.418,-1.206,-2.459,-0.060,-1.161,-1.164
1803,2025-09-18,594.631,-1.305,-1.360,-2.472,-1.186,-2.459,-1.103,-2.341,-2.243
1802,2025-09-17,589.317,-2.378,-1.379,-2.398,-1.214,-2.481,-1.101,-2.351,-2.178
1801,2025-09-16,590.495,-1.364,-1.350,-2.415,-1.121,-2.481,2.018,-1.172,-1.080
1800,2025-09-15,590.995,-2.395,-2.388,-2.388,-0.048,-2.479,-0.074,-1.226,-1.121
1799,2025-09-12,585.981,-1.344,-2.436,-2.365,-1.096,-2.465,0.951,-1.289,-0.095
1798,2025-09-11,583.404,-1.293,-2.390,-2.258,-1.151,-2.463,-0.060,-2.438,-0.093


In [108]:
def performance_metrics(metrics_df, returns):

    metrics_df = metrics_df.sort_index(ascending=True)

    def add_column_based_on_future_value(df, days):

        future_return = (df['Close'].shift(-days) - df['Close']) / df['Close']

        if days >= 100:
            df[f'Return_{days}'] = np.where(
                future_return > 0.001, 1,
                np.where(future_return < -0.001, 0, np.nan)
            )
        else:
            df[f'Return_{days}'] = (future_return > 0).astype(int)

        return df

    # Apply return logic for each target horizon
    for r in returns: 
        new_df = add_column_based_on_future_value(metrics_df, r)

    return new_df

performance_df = performance_metrics(sub_model_df.round(3), returns)

horizons = returns
# Dictionary to store the filtered DataFrames
horizon_dfs = {}

# Loop through each horizon
for h in horizons:
    # Collect relevant columns
    cols = ['Date', 'Close'] + [
        col for col in performance_df.columns 
        if col.endswith(f'_xgb_{h}deep') or col.endswith(f'_xgb_{h}shallow') or col == f'sum_{h}' or col == f'Return_{h}'
    ]
    
    # Create and store the filtered DataFrame
    horizon_dfs[h] = performance_df[cols].copy()

def convert_signed_to_prob(p):
    return p if p >= 0 else 1 + p

for h in horizons:
    df = horizon_dfs[h].sort_values('Date', ascending=False).iloc[h:h+100, :].copy()
    
    for arch in ['deep']:
        records = []
        actual_pos = df[f'Return_{h}'].sum()
        actual_neg = len(df[f'Return_{h}']) - actual_pos
        for col in df.columns:
            if f'_xgb_{h}{arch}' in col:
                # only keep the rows that actually got a prediction
                filtered = df[df[col] != 0]
                # convert your raw score to a probability
                probs = filtered[col].apply(convert_signed_to_prob)
                # binarize at 0.5
                y_pred = (probs >= 0.5).astype(int)
                y_true = filtered[f'Return_{h}']
                
                # compute the matrix
                cm = confusion_matrix(y_true, y_pred,labels=[0,1])
                tn, fp, fn, tp = cm.ravel()
                records.append({
                    'model': col,
                    'act_pos': actual_pos,
                    'act_neg': actual_neg,
                    'pos_acc': f"{(tp/(tp + fp)):.1%}" if actual_pos>0 else "n/a",
                    'neg_acc': f"{(tn/(tn + fn)):.1%}" if actual_neg>0 else "n/a",
                    'tp': tp,
                    'fp': fp,
                    'tn': tn,
                    'fn': fn,
                })

        # make your summary table
        cm_df = pd.DataFrame(records)
        print(cm_df.to_string())
        print(' ')

                                                                             model  act_pos  act_neg pos_acc neg_acc  tp  fp  tn  fn
0                                 raw_trend_fast+raw_trend_ratio_slow_25_xgb_2deep       63       37   70.1%   51.5%  47  20  17  16
1                 raw_trend_ratio_moderate_25+raw_volatility_moderate_25_xgb_2deep       63       37   68.4%   57.1%  54  25  12   9
2  raw_trend_ratio_fast_25+raw_momentum_moderate_10+raw_momentum_fast_10_xgb_2deep       63       37   74.0%   66.7%  54  19  18   9
3                   raw_trend_ratio_moderate_10+raw_momentum_moderate_25_xgb_2deep       63       37   69.3%   56.0%  52  23  14  11
 
                                                                             model  act_pos  act_neg pos_acc neg_acc  tp  fp  tn  fn
0  raw_trend_ratio_moderate_25+raw_volatility_moderate_25+raw_trend_fast_xgb_3deep       72       28   84.0%   78.9%  68  13  15   4
1        raw_trend_fast+raw_trend_ratio_slow_25+raw_trend_ratio_mod